# DreaMS embeddings and molecular families

**Context.** PRIORITY project (WUR). Meeting with Joana Tirana and Daniel Probst on 19 Oct 2026: embeddings for metabolomics, molecular families, and the DreaMS paper (Bushuiev et al., *Nature Biotechnology*, 2025).
**Author:** Vitória Maia

**Questions.** Do DreaMS embeddings of MS/MS spectra put metabolites of the same molecular family close together? Does DreaMS similarity follow chemical structure better than the modified cosine that GNPS uses, and better than MS2DeepScore? As a first look towards federated learning: how different are the labs, and does a small layer trained across labs help?

**Steps**
0. Check that our DreaMS installation reproduces the numbers of the DreaMS networking tutorial.
1. Take six public GNPS molecular networking jobs on actinobacteria registered in the Paired Omics Data Platform.
2. Keep the spectra that pass DreaMS's own format A check, unchanged (the *protocol* variant). Our earlier cleaning is kept as a sensitivity analysis (*all_kept*).
3. Compute embeddings with the **pre-trained** DreaMS and MS2DeepScore models.
4. **Experiment A:** neighbours and a DreaMS network vs the GNPS molecular families.
5. **Experiment B:** spectral similarities vs structural similarity (Morgan Tanimoto and MHFP) on library-annotated spectra.
6. **Experiment C (prototype):** a small layer on frozen DreaMS embeddings, trained locally, centrally, with FedAvg and SplitFedV2-like, one lab per client.
7. **Experiment D:** can a classifier tell the labs apart from their embeddings?
8. Which public datasets could test DreaMS for genome-metabolome linking?

Each step is a script in `scripts/`. The notebook runs them and shows the results. `RESULTS.md` has the numbers of A and B in one page.

## 0. Setup

**One-time steps** (in a terminal, inside `priority-wur/dreams_molecular_families`). DreaMS and MS2DeepScore pin different package versions, so each gets its own Python 3.11 environment, made with [uv](https://docs.astral.sh/uv/):
```
git clone https://github.com/pluskal-lab/DreaMS.git .DreaMS
git -C .DreaMS checkout dbec3a0b514a99e5056cfccde4559fda8cfe8129
uv venv .dreams_env --python 3.11
uv pip install --python .dreams_env/Scripts/python.exe -e ./.DreaMS mhfp==1.9.6 ipykernel nbclient
uv venv .ms2ds_env --python 3.11
uv pip install --python .ms2ds_env/Scripts/python.exe "ms2deepscore>=2.6" onnxruntime
```
(On Mac/Linux, use `bin/python` instead of `Scripts/python.exe`.) The first runs download the DreaMS weights (2.6 GB, Hugging Face) and the MS2DeepScore model (416 MB, Zenodo). These folders are git-ignored.

**Data.** The GNPS jobs come from `data/raw/gnps/`, which `python -I scripts/download_data.py gnps` fills (run from the repository root).

**Kernel.** Any Jupyter kernel with pandas works: the steps always run with the Python of the right environment.

In [ ]:
import importlib.util
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

RUNS = ["duncan_salinispora", "crusemann_salstrep", "crusemann_streptomyces",
        "chevrette_insect", "dorrestein_salbus", "tahlan_streptomyces"]
VARIANTS = ["protocol", "all_kept"]
RESULTS_DIR = {"protocol": "results", "all_kept": "results_all_kept"}

EXPERIMENT_DIR = Path.cwd()
if not (EXPERIMENT_DIR / "scripts" / "common.py").exists():
    raise RuntimeError("Open this notebook from the dreams_molecular_families folder.")


def env_python(name):
    path = EXPERIMENT_DIR / name / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    if not path.exists():
        raise RuntimeError(f"{name} not found. Do the one-time steps in section 0.")
    return str(path)


# Python used for the steps: this kernel if it has DreaMS, otherwise .dreams_env.
PY = sys.executable if importlib.util.find_spec("dreams") is not None else env_python(".dreams_env")
PY_MS2DS = env_python(".ms2ds_env")
print("DreaMS steps:", PY)
print("MS2DeepScore steps:", PY_MS2DS)


def step(script, *args, python=None, show=False, allow_fail=False):
    # Run one script and stop with its error message if it fails.
    start = time.time()
    result = subprocess.run(
        [python or PY, f"scripts/{script}", *args], capture_output=True, text=True,
        encoding="utf-8", errors="replace", env={**os.environ, "PYTHONUTF8": "1"},
    )
    if result.returncode != 0:
        if allow_fail:
            print(f"skipped: {script} {' '.join(args)}: {result.stdout.strip().splitlines()[-1:] or result.stderr[-300:]}")
            return
        print(result.stdout[-5000:], result.stderr[-5000:], sep="\n")
        raise RuntimeError(f"{script} {' '.join(args)} failed. Read the error above.")
    if show:
        print(result.stdout)
    print(f"done: {script} {' '.join(args)} ({time.time() - start:.0f} s)")


def results(run, name, variant="protocol"):
    return pd.read_parquet(EXPERIMENT_DIR / "runs" / run / RESULTS_DIR[variant] / f"{name}.parquet")


def side_by_side(name, key="metric", value="value", variant="protocol"):
    # One summary table per run, merged into one table with a column per run.
    tables = []
    for run in RUNS:
        path = EXPERIMENT_DIR / "runs" / run / RESULTS_DIR[variant] / f"{name}.parquet"
        if path.exists():
            tables.append(pd.read_parquet(path).set_index(key)[value].rename(run))
    return pd.concat(tables, axis=1).fillna("")


pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)

## 0b. Does our DreaMS installation reproduce a published result?

Before comparing DreaMS with other methods, we check our setup against the DreaMS networking tutorial, which the DreaMS authors ran and saved with its outputs. It uses one public file (S_N1.mzML, MassIVE MSV000086206) and reports 3,809 spectra, a DreaMS similarity of 0.9109248 between spectra 1137 and 1165, and 6,858 edges in the DreaMS k-NN network. The script follows the tutorial code exactly, so none of our own processing is involved.

In [ ]:
reproduction = EXPERIMENT_DIR / "reproduction" / "dreams_networking_tutorial.json"
if not reproduction.exists():
    step("reproduce_dreams_tutorial.py")
display(pd.read_json(reproduction, typ="series"))

## 1. Data and the two versions of the input spectra

| Run | Dataset | MassIVE (PoDP) | GNPS fragment tolerance |
|---|---|---|---|
| `duncan_salinispora` | *Salinispora*, Duncan et al. 2015 (ion trap). Example job of the NPLinker docs | MSV000079284 | 0.3 Da |
| `crusemann_salstrep` | *Salinispora* + *Streptomyces*, Crüsemann et al. 2017 | MSV000078836 | 0.5 Da |
| `crusemann_streptomyces` | *Streptomyces*, Crüsemann et al. 2017 | MSV000078839 | 0.5 Da |
| `chevrette_insect` | Insect-associated actinobacteria, Chevrette et al. 2019 | MSV000084723 | 0.02 Da |
| `dorrestein_salbus` | *Streptomyces albus* J1074, Dorrestein lab | MSV000079519 | 0.01 Da |
| `tahlan_streptomyces` | Clavulanic acid producing *Streptomyces*, Tahlan lab | MSV000083835 | 0.02 Da |

All are GNPS jobs that the Paired Omics Data Platform links to genomes. The two Crüsemann jobs overlap: the SalStrep job also re-analyses MSV000078839.

**Molecular families.** GNPS connects two spectra when their modified cosine is high enough. Each connected group is a molecular family (`componentindex`). Spectra without connections are singletons.

**Checks.** `prepare_data.py` stops if the spectra with peaks are not exactly the network nodes, or if edges or library hits point to unknown nodes.

**Two versions of the input spectra.**
- `protocol` (main): only the spectra that pass DreaMS's own format A check (`DataFormatA().val_spec`: charge +1, precursor and fragments at most m/z 1000, 3 to 128 peaks, at least 3 peaks above 10% intensity, max/min intensity at least 20), given to every method exactly as GNPS gives them.
- `all_kept` (sensitivity analysis): our first cleaning. It removes the peaks above the precursor *before* applying the same rules, so it also keeps spectra that DreaMS's own check rejects: old ion trap data in the Duncan job (fragments heavier than the precursor, which is noise), and noise above m/z 1000 in every spectrum of the Crüsemann *Streptomyces* job. That job has no `protocol` spectra.

In [ ]:
for run in RUNS:
    step("prepare_data.py", "--run", run)

info = pd.DataFrame({run: pd.read_json(EXPERIMENT_DIR / "runs" / run / "results" / "run_info.json", typ="series") for run in RUNS})
display(info.drop([i for i in info.index if "md5" in i or i == "run"]))
display(side_by_side("00_filter_report", key="step", value="spectra"))

## 2. Embeddings

We use the pre-trained `DreaMS_embedding` model (one vector of 1024 numbers per spectrum) and, as a baseline, the pre-trained MS2DeepScore model recommended by its authors (van der Hooft lab, WUR; trained to predict structural similarity from > 500,000 spectra). Spectra that a model sees as similar get vectors that point in a similar direction (high cosine similarity).

On a laptop CPU DreaMS embeds about 2 to 4 spectra per second; MS2DeepScore is much faster. Embeddings are saved in `runs/<run>/data/` and only computed again if the input spectra change.

In [ ]:
for variant in VARIANTS:
    for run in RUNS:
        step("embed_dreams.py", "--run", run, "--variant", variant, allow_fail=True)
        step("embed_ms2deepscore.py", "--run", run, "--variant", variant, python=PY_MS2DS, allow_fail=True)

embeddings = {}
for run in RUNS:
    path = EXPERIMENT_DIR / "runs" / run / "data" / "dreams_embeddings_protocol.json"
    if path.exists():
        embeddings[run] = pd.read_json(path, typ="series")
display(pd.DataFrame(embeddings))

## 3. Experiment A: molecular families

- **A1. Neighbours.** For each spectrum in a GNPS family with at least 3 members, we look at its 5 nearest neighbours. *Family recovery@5* = family partners found, divided by the best possible number. Compared with MS2DeepScore, the cosine of binned spectra (1 Da bins, no learning), neighbours chosen only by precursor m/z, and random neighbours. Differences have a 95% bootstrap interval that resamples whole GNPS families; an interval that contains 0 means no detectable difference.
- **A2. GNPS edges.** Pairs connected in the GNPS network should have a higher DreaMS similarity than random pairs.
- **A3. DreaMS network.** Each spectrum is linked to its 3 nearest neighbours if the DreaMS similarity is at least 0.7 (as in the DreaMS networking tutorial). We compare the resulting families with the GNPS families (AMI/ARI). DreaMS links that involve GNPS singletons are saved in `results/exp_a_new_links.parquet`: hypotheses to check by hand, not results.
- **A4. Network thresholds.** The same DreaMS network with thresholds from 0.6 to 0.95.

⚠️ GNPS families come from modified cosine, so this experiment measures **agreement with GNPS**, not truth. Experiment B uses chemical structures instead.

In [ ]:
for variant in VARIANTS:
    for run in RUNS:
        step("exp_a_families.py", "--run", run, "--variant", variant, allow_fail=True)

display(Markdown("**Protocol (main)**"))
display(side_by_side("exp_a_summary"))
display(Markdown("**Our cleaning (sensitivity analysis)**"))
display(side_by_side("exp_a_summary", variant="all_kept"))

In [ ]:
for run in RUNS:
    folder = EXPERIMENT_DIR / "runs" / run / "results"
    if (folder / "exp_a_umap.png").exists():
        display(Markdown(f"### `{run}` (protocol)"))
        display(Image(folder / "exp_a_umap.png", width=560))
        display(Image(folder / "exp_a_similarity_hist.png", width=460))

In [ ]:
# DreaMS links that GNPS did not make (first rows). Look at the delta m/z
# (for example +14 = CH2, +16 = O) and at the library names.
for run in RUNS:
    path = EXPERIMENT_DIR / "runs" / run / "results" / "exp_a_new_links.parquet"
    if path.exists():
        display(Markdown(f"**New links: `{run}`**"))
        display(pd.read_parquet(path).head(10))

## 4. Experiment B: chemical structures

Only spectra with a GNPS library hit that has a structure (SMILES). For every pair, the structural similarity of the two molecules (Tanimoto on Morgan fingerprints; also MHFP, Daniel Probst's MinHash fingerprint) is compared with DreaMS, MS2DeepScore, modified cosine (same fragment tolerance as the GNPS job) and binned spectrum cosine.

- **B1.** Spearman correlation over all pairs. The difference between DreaMS and the other methods gets a 95% interval from resampling molecules.
- **B2.** For each spectrum, take its most similar other spectrum and check the Tanimoto of the two molecules. *Analog search* skips spectra of the same molecule.
- A dataset counts for conclusions only with at least 100 different molecules (`B used for conclusions`). We set this rule after the first results, when no method followed the 77 molecules of the Duncan job.

In [ ]:
for variant in VARIANTS:
    for run in RUNS:
        step("exp_b_structures.py", "--run", run, "--variant", variant, allow_fail=True)

display(side_by_side("exp_b_summary"))
for run in RUNS:
    figure = EXPERIMENT_DIR / "runs" / run / "results" / "exp_b_structure.png"
    if figure.exists():
        display(Image(figure, width=1000))

## 5. Experiment C (prototype): a small layer trained across labs

Each lab is one client. A linear layer on top of the frozen DreaMS embeddings is trained so that the cosine of two spectra follows the Tanimoto of their molecules (the training target of MS2DeepScore). Compared: pre-trained DreaMS, local training, centralised training (all spectra in one place), a control with only pairs from one lab, FedAvg, and SplitFedV2-like split learning (the server trains on one lab after the other). With DreaMS frozen on the client side, split learning in which the server may pair spectra from different labs is the same as centralised training on the embeddings.

*Performance gap between labs* is the metric of Tirana et al. (AAAI 2026, `my_utils.performance_gap`), over labs instead of labels: 0 = all labs equally well served.

In [ ]:
for variant in VARIANTS:
    step("exp_c_federated_adapter.py", "--variant", variant)
display(Markdown((EXPERIMENT_DIR / "federated_prototype" / "RESULTS_federated_prototype.md").read_text(encoding="utf-8")))

## 6. Experiment D: how different are the labs?

A federated study needs a measured heterogeneity. Can a logistic regression tell two labs apart from their embeddings? Balanced accuracy 0.5 = the labs look the same, 1.0 = fully separated. Labs differ in chemistry and in measurement at the same time; the test on molecules that both labs measured holds the chemistry fixed.

In [ ]:
step("exp_d_lab_shift.py")
display(Markdown((EXPERIMENT_DIR / "lab_shift" / "RESULTS_lab_shift_protocol.md").read_text(encoding="utf-8")))

## 7. Could DreaMS be tested for genome-metabolome linking?

NPLinker-style linking connects molecular families to gene cluster families through their presence across strains, and is judged on known BGC-spectrum links. A fair test of DreaMS for this needs many strains with genomes, known links, and spectra that DreaMS accepts.

In [ ]:
step("linking_feasibility.py")
display(Markdown((EXPERIMENT_DIR / "linking_feasibility" / "RESULTS_linking_feasibility.md").read_text(encoding="utf-8")))

## 8. All results in one page, and automatic checks

`scripts/report.py` writes `RESULTS.md`. `scripts/check_pipeline.py` runs unit checks, checks that every file matches its spectra, and compares the summaries with the saved reference in `tests/reference/`.

In [ ]:
step("report.py", "--runs", *RUNS)
step("check_pipeline.py", show=True)

## 9. How to read the results

Write your own interpretation here before the meeting. Some guiding questions:
- Does our setup reproduce the DreaMS tutorial (section 0b)? If not, nothing below can be trusted yet.
- In A1, do the 95% intervals of *DreaMS minus binned cosine* and *DreaMS minus MS2DeepScore* contain 0? Is it the same in all datasets?
- In B, which datasets count for conclusions? There, is DreaMS better than modified cosine and MS2DeepScore, or not detectably different?
- How do the `protocol` and `all_kept` results differ, especially for Duncan?
- In C, does any trained layer beat pre-trained DreaMS on pairs from two labs? Is there a performance gap between labs?
- In D, are the labs different only in chemistry, or also in measurement?
- Which public dataset, if any, could test DreaMS for linking?

**My interpretation:** *(write here)*

## 10. Limitations

1. **GNPS families are not ground truth.** They come from modified cosine, so Experiment A measures agreement with GNPS.
2. **Library hits are putative.** They are spectral matches, not confirmed structures. Only datasets with at least 100 different molecules count for conclusions in Experiment B.
3. **Consensus spectra.** GNPS gives clustered (consensus) spectra; DreaMS was trained on single scans. The `protocol` variant follows DreaMS's quality rules but still uses consensus spectra.
4. **Possible overlap with training data.** These datasets are public on MassIVE, so they may be part of GeMS (DreaMS pre-training, without labels). The DreaMS embedding model and MS2DeepScore were trained on public library spectra with known structures; the library spectra behind our annotations may be among them, which can favour both models in Experiment B.
5. **The runs are not independent.** The SalStrep job also re-analyses the *Streptomyces* data of MSV000078839.
6. **Small numbers.** Few annotated spectra in most datasets; in Experiment C one lab has most of the training spectra.
7. **Experiments C and D are prototypes**, simulated on one machine, with settings fixed in advance and not tuned.

## 11. Open questions

**For Joana and Daniel**
1. What did "train" mean in our meeting: fine-tuning DreaMS, training a new model, or federated training?
2. Partner spectra will mostly have no annotation. Should a federated version be self-supervised (like DreaMS pre-training)? That needs a GPU: is there a WUR server I can use?
3. Is there a high-resolution actinobacteria dataset in PRIORITY with many strains and known links, so that DreaMS can be tested for linking?
4. Is the data-quality finding (the first "difference between labs" was out-of-domain data) interesting for federated learning, for example as a check before a client joins?